# KTO: Kahneman-Tversky Optimization (2024)

## The Problem with DPO and ORPO

DPO and ORPO both require **paired** preference data: for each prompt, you need a (chosen, rejected) pair.

In production this is expensive. But you already have binary signals from users: thumbs up/down, 👍/👎, ratings, regenerations. These aren't paired.

**KTO** (Ethayarajh et al. 2024) solves this. It takes inspiration from **Prospect Theory** (Kahneman & Tversky's Nobel Prize-winning behavioral economics work): humans are loss-averse — the pain of a loss is roughly twice the pleasure of an equivalent gain.

KTO loss:
```
L_KTO = E[w(y) · (1 - v(x, y))]

where:
  v(x,y)   = σ(β · (log π(y|x)/π_ref(y|x) - z₀))    # individual value
  z₀       = E_y~ref[log π(y|x)/π_ref(y|x)]           # KL reference point
  w(y)     = λ_D if y is desirable, λ_U if undesirable  # asymmetric weights
```

The loss-aversion asymmetry means KTO naturally handles imbalanced feedback.

In [ ]:
!pip install -U trl transformers datasets peft -q
import trl; print(f"TRL {trl.__version__}")

In [ ]:
import torch, gc
from trl.experimental.kto import KTOConfig, KTOTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
from peft import LoraConfig

gc.collect(); torch.cuda.empty_cache()

model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto")
print(f"Model: {sum(p.numel() for p in model.parameters())/1e6:.0f}M params")
print(f"Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## KTO Dataset Format

Unlike DPO, KTO takes **individual** examples with binary labels — no pairing required.
This mirrors real production feedback: every user interaction generates a signal.

In [ ]:
# KTO format: {prompt, completion, label}
# label=True  → desirable (thumbs up, accepted, not regenerated)
# label=False → undesirable (thumbs down, rejected, regenerated)

kto_data = [
    # Desirable (high-quality) responses
    {"prompt": "What is LoRA?",
     "completion": "LoRA adds trainable low-rank matrices to frozen weights, reducing parameters by 99% while maintaining fine-tuning quality.",
     "label": True},
    {"prompt": "Explain gradient descent.",
     "completion": "Gradient descent minimizes loss by iteratively updating θ ← θ - α∇L(θ). The learning rate α controls step size.",
     "label": True},
    {"prompt": "What is the attention mechanism?",
     "completion": "Attention computes Softmax(QK^T/√d_k)·V. Queries, keys, and values allow each token to attend to all others.",
     "label": True},
    {"prompt": "What is fine-tuning?",
     "completion": "Fine-tuning adapts a pre-trained model to a specific task by continuing training on domain-specific data with a small learning rate.",
     "label": True},

    # Undesirable (low-quality) responses
    {"prompt": "What is LoRA?",
     "completion": "LoRA is a popular Japanese animation series featuring magical characters.",
     "label": False},
    {"prompt": "Explain gradient descent.",
     "completion": "Gradient descent is a ski technique for safely descending steep mountain slopes.",
     "label": False},
    {"prompt": "What is the attention mechanism?",
     "completion": "Attention means paying close attention to what your teacher says in class.",
     "label": False},
    {"prompt": "What is fine-tuning?",
     "completion": "Fine-tuning is when you adjust the settings on a radio to get better reception.",
     "label": False},
] * 4  # 32 total, 16 positive + 16 negative

dataset = Dataset.from_list(kto_data)
print(f"KTO dataset: {len(dataset)} individual examples")
print(f"  Desirable: {sum(d['label'] for d in kto_data)}")
print(f"  Undesirable: {sum(not d['label'] for d in kto_data)}")
print()
print("Critically: these are NOT pairs — any completion can appear independently")

In [ ]:
config = KTOConfig(
    output_dir="/tmp/kto",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-7,
    beta=0.1,                    # KL penalty (same role as DPO beta)
    max_length=256,
    desirable_weight=1.0,        # λ_D in paper — weight for positive examples
    undesirable_weight=1.0,      # λ_U in paper — can set higher for loss-aversion
    remove_unused_columns=False,
    fp16=True,
    logging_steps=4,
    save_strategy="no",
    report_to="none",
)

lora_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

trainer = KTOTrainer(
    model=model,
    ref_model=None,          # uses implicit reference (PEFT adapter)
    args=config,
    train_dataset=dataset,
    peft_config=lora_config,
)

result = trainer.train()
print(f"\n✅ KTO training complete!")
print(f"   Loss: {result.training_loss:.4f}")
print(f"   Runtime: {result.metrics['train_runtime']:.1f}s")
print(f"   No paired data needed — works with any binary feedback!")

## KTO vs DPO: When to Use Each

| | DPO | ORPO | KTO |
|---|---|---|---|
| Data format | Paired (chosen + rejected) | Paired (chosen + rejected) | Individual + binary label |
| Reference model | Required | Not needed | Can omit with PEFT |
| Data collection | Hard (need pairs) | Hard (need pairs) | Easy (thumbs up/down) |
| Memory | 2× model | 1× model | 2× model |
| Quality vs DPO | Comparable | Comparable | Comparable at 1-30B |
| Best for | Offline annotation | Memory-constrained | Production feedback loops |

**Rule of thumb**: If you're deploying a model and collecting feedback from real users, use KTO.
If you're using human annotators to create preference pairs, use DPO or ORPO.